In [1]:
import sys
sys.path.insert(0, '.')
import pandas as pd
from src.parser import parse_deck_list
from src.api_client import enrich_deck,_fetch_by_name
from src.deck import Deck
from src import calculator as calc
from src import monte_carlo as mc


In [2]:
# DECK_LIST = """
# Pokémon: 20
# 1 Alakazam MEP 9
# 2 Alakazam MEG 56
# 4 Kadabra MEG 55
# 4 Abra MEG 54
# 3 Dudunsparce PRE 80
# 2 Dunsparce JTG 120
# 1 Dunsparce TEF 128
# 1 Psyduck MEP 7
# 1 Fezandipiti ex SFA 38
# 1 Shaymin DRI 10

# Trainer: 33
# 1 Boss's Orders ASC 256
# 1 Boss's Orders PAL 265
# 1 Boss's Orders MEG 114
# 3 Rare Candy MEG 175
# 2 Battle Cage PFL 116
# 2 Battle Cage PFL 85
# 1 Sacred Ash POR 115
# 1 Eri TEF 146
# 3 Buddy-Buddy Poffin TEF 144
# 1 Buddy-Buddy Poffin ASC 184
# 2 Enhanced Hammer TWM 224
# 3 Poké Pad POR 81
# 1 Poké Pad POR 113
# 1 Lana's Aid TWM 155
# 4 Hilda WHT 84
# 4 Dawn PFL 87
# 1 Night Stretcher SFA 61
# 1 Wondrous Patch POR 117

# Energy: 7
# 1 Enriching Energy SSP 191
# 4 Telepathic Psychic Energy POR 88
# 2 Psychic Energy MEE 5
# """


In [3]:
# DECK_LIST = """
# Pokémon: 20
# 4 N's Zorua JTG 97
# 4 N's Zoroark ex JTG 98
# 2 N's Darumaka JTG 26
# 2 N's Darmanitan JTG 27
# 2 N's Zekrom ASC 155
# 1 N's Reshiram JTG 116
# 1 Budew ASC 16
# 1 Munkidori TWM 95
# 1 Yveltal MEG 88
# 1 Pecharunt ex SFA 39
# 1 Meowth ex POR 62

# Trainer: 32
# 4 Lillie's Determination MEG 119
# 3 Cyrano SSP 170
# 3 Boss's Orders MEG 114
# 1 Janine's Secret Art PRE 112
# 1 Black Belt's Training JTG 143
# 4 Ultra Ball MEG 131
# 4 Buddy-Buddy Poffin TEF 144
# 3 Poké Pad ASC 198
# 3 N's PP Up JTG 153
# 2 Night Stretcher ASC 196
# 1 Unfair Stamp TWM 165
# 1 Binding Mochi PRE 95
# 2 N's Castle JTG 152

# Energy: 8
# 8 Darkness Energy MEE 7
# """

In [4]:
# DECK_LIST ="""
# Pokémon: 17
# 4 Riolu MEG 76
# 3 Mega Lucario ex MEG 77
# 3 Solrock MEG 75
# 2 Lunatone MEG 74
# 2 Makuhita MEG 72
# 2 Hariyama MEG 73
# 1 Meowth ex POR 62

# Trainer: 31
# 4 Lillie's Determination MEG 119
# 2 Judge DRI 167
# 2 Boss's Orders MEG 114
# 2 Wally's Compassion MEG 132
# 4 Ultra Ball MEG 131
# 4 Fighting Gong MEG 116
# 4 Premium Power Pro MEG 124
# 3 Poké Pad ASC 198
# 2 Switch MEG 130
# 1 Air Balloon ASC 181
# 1 Maximum Belt TEF 154
# 2 Gravity Mountain SSP 177

# Energy: 12
# 10 Fighting Energy MEE 6
# 2 Rocky Fighting Energy POR 87
# """

In [5]:
DECK_LIST ="""
Pokémon: 10
4 Dwebble DRI 11
3 Crustle DRI 12
2 Mega Kangaskhan ex MEG 104
1 Cornerstone Mask Ogerpon ex TWM 112

Trainer: 35
4 Lillie's Determination MEG 119
4 Team Rocket's Petrel DRI 176
4 Boss's Orders MEG 114
2 Hilda WHT 84
2 Colress's Tenacity SFA 57
1 Xerosic's Machinations SFA 64
1 Pokémon Center Lady MEG 123
4 Pokégear 3.0 SVI 186
4 Jumbo Ice Cream PFL 91
2 Buddy-Buddy Poffin TEF 144
2 Ultra Ball MEG 131
1 Switch MEG 130
1 Super Potion JTG 158
1 Hero's Cape TEF 152
1 Team Rocket's Factory DRI 173
1 Forest of Vitality MEG 117

Energy: 15
4 Growing Grass Energy POR 86
4 Mist Energy TEF 161
4 Spiky Energy JTG 159
2 Rocky Fighting Energy POR 87
1 Grass Energy MEE 1
"""

In [6]:
# Optional: customize the target cards and only the searches that can find them
TARGET_CARD_NAMES = ["N's Zorua"]
TARGET_SEARCH_NAMES = [
    "Buddy-Buddy Poffin",
    "Poké Pad",
    "Ultra Ball"
    ]

# Monte Carlo settings
MC_SIMULATIONS = 100_000
MC_SEED = 42

In [7]:
print("Parsing deck list...")
parsed = parse_deck_list(DECK_LIST)
print(f"Found {len(parsed)} unique card entries. Looking up via TCGDex API (uses cache)...")
cards = enrich_deck(parsed, cache_path="card_cache.json")
deck = Deck(cards)
print(f"Total cards: {deck.total_cards} | Basic Pokemon: {deck.total_basics}")
unknown = [c for c in deck.cards if c.subcategory == 'unknown']
if unknown:
    print(f"WARNING: {len(unknown)} cards not found in API: {[c.name for c in unknown]}")
else:
    print("All cards classified successfully.")


Parsing deck list...
Found 25 unique card entries. Looking up via TCGDex API (uses cache)...
Total cards: 60 | Basic Pokemon: 7
All cards classified successfully.


In [8]:
print("=" * 50)
print("DECK BREAKDOWN")
print("=" * 50)

breakdown_data = {
    "Category": ["Pokémon", "", "Trainer", "", "", "", "Energy", ""],
    "Subcategory": ["Basic", "Other", "Item", "Supporter", "Stadium", "Tool", "Basic", "Special"],
    "#": [
        sum(c.quantity for c in deck.cards if c.subcategory == "basic"),
        sum(c.quantity for c in deck.cards if c.subcategory == "other"),
        deck.trainers_by_subtype.get("item", 0),
        deck.trainers_by_subtype.get("supporter", 0),
        deck.trainers_by_subtype.get("stadium", 0),
        deck.trainers_by_subtype.get("tool", 0),
        deck.energies_by_subtype.get("basic_energy", 0),
        deck.energies_by_subtype.get("special_energy", 0),
    ]
}
df_breakdown = pd.DataFrame(breakdown_data)
display(df_breakdown)
print(f"Total: {deck.total_cards} cards")


DECK BREAKDOWN


,Category,Subcategory,#
0,Pokémon,Basic,7
1,,Other,3
2,Trainer,Item,14
3,,Supporter,18
4,,Stadium,2
5,,Tool,1
6,Energy,Basic,7
7,,Special,8


Total: 60 cards


In [9]:
print("=" * 50)
print("OPENING HAND PROBABILITIES")
print("=" * 50)

N = deck.total_cards
b = deck.total_basics

opening_data = {
    "Event": ["Mulligan (no Basic)", "Starting with exactly 1 Basic", "Starting with 2 or more Basics"],
    "Probability": [
        f"{calc.mulligan_probability(N, b):.2%}",
        f"{calc.exactly_one_basic_probability(N, b):.2%}",
        f"{calc.two_or_more_basics_probability(N, b):.2%}",
    ]
}
display(pd.DataFrame(opening_data))


OPENING HAND PROBABILITIES


,Event,Probability
0,Mulligan (no Basic),39.91%
1,Starting with exactly 1 Basic,69.25%
2,Starting with 2 or more Basics,30.75%


In [10]:
print("=" * 50)
print("STARTER PROBABILITIES (per Basic Pokémon)")
print("=" * 50)

starter_data = []
for card in deck.basic_pokemon:
    possible = calc.possible_starter_probability(N, b, card.quantity)
    forced = calc.forced_starter_probability(N, b, card.quantity)
    starter_data.append({
        "Pokémon": f"{card.name} ({card.set_code} #{card.set_number})",
        "Copies": card.quantity,
        "Possible Starter": f"{possible:.2%}",
        "Forced Starter": f"{forced:.2%}",
    })
display(pd.DataFrame(starter_data))


STARTER PROBABILITIES (per Basic Pokémon)


,Pokémon,Copies,Possible Starter,Forced Starter
0,Dwebble (DRI #11),4,66.49%,47.51%
1,Mega Kangaskhan ex (MEG #104),2,36.86%,21.02%
2,Cornerstone Mask Ogerpon ex (TWM #112),1,19.42%,9.89%


In [11]:
print("=" * 50)
print("PRIZE CARD PROBABILITIES")
print("=" * 50)

prize_data = []
for card in deck.cards:
    prob = calc.prize_probability(N, card.quantity)
    prize_data.append({
        "Card": card.name,
        "Copies": card.quantity,
        "P(≥1 Prized)": f"{prob:.2%}",
    })
df_prize = pd.DataFrame(prize_data).sort_values("P(≥1 Prized)", ascending=False)
display(df_prize.reset_index(drop=True))


PRIZE CARD PROBABILITIES


,Card,Copies,P(≥1 Prized)
0,Dwebble,4,35.15%
1,Boss's Orders,4,35.15%
2,Spiky Energy,4,35.15%
3,Mist Energy,4,35.15%
4,Growing Grass Energy,4,35.15%
5,Pokégear 3.0,4,35.15%
6,Jumbo Ice Cream,4,35.15%
7,Team Rocket's Petrel,4,35.15%
8,Lillie's Determination,4,35.15%
9,Crustle,3,27.52%


In [12]:
print("=" * 50)
print("DRAW BY TURN")
print("=" * 50)

MAX_TURNS = 6
draw_data = []
for card in deck.cards:
    if card.quantity == 0:
        continue
    row = {"Card": card.name, "Copies": card.quantity}
    for t in range(1, MAX_TURNS + 1):
        row[f"Turn {t}"] = f"{calc.draw_by_turn_probability(N, card.quantity, t):.2%}"
    draw_data.append(row)
display(pd.DataFrame(draw_data))


DRAW BY TURN


,Card,Copies,Turn 1,Turn 2,Turn 3,Turn 4,Turn 5,Turn 6
0,Dwebble,4,39.95%,44.48%,48.75%,52.77%,56.55%,60.10%
1,Crustle,3,31.54%,35.42%,39.14%,42.72%,46.16%,49.46%
2,Mega Kangaskhan ex,2,22.15%,25.08%,27.97%,30.79%,33.56%,36.27%
3,Cornerstone Mask Ogerpon ex,1,11.67%,13.33%,15.00%,16.67%,18.33%,20.00%
4,Lillie's Determination,4,39.95%,44.48%,48.75%,52.77%,56.55%,60.10%
5,Team Rocket's Petrel,4,39.95%,44.48%,48.75%,52.77%,56.55%,60.10%
6,Boss's Orders,4,39.95%,44.48%,48.75%,52.77%,56.55%,60.10%
7,Hilda,2,22.15%,25.08%,27.97%,30.79%,33.56%,36.27%
8,Colress's Tenacity,2,22.15%,25.08%,27.97%,30.79%,33.56%,36.27%
9,Xerosic's Machinations,1,11.67%,13.33%,15.00%,16.67%,18.33%,20.00%


In [13]:
print("=" * 50)
print("SUPPORTER & DEAD HAND")
print("=" * 50)

s = deck.trainers_by_subtype.get("supporter", 0)
e = sum(deck.energies_by_subtype.values())

support_data = {
    "Statistic": [
        "Supporter in opening hand",
        "Dead Hand (0 Supporter + 0 Energy)",
    ],
    "Probability": [
        f"{calc.supporter_turn1_probability(N, s):.2%}",
        f"{calc.dead_hand_probability(N, s, e):.2%}",
    ]
}
display(pd.DataFrame(support_data))


SUPPORTER & DEAD HAND


,Statistic,Probability
0,Supporter in opening hand,93.01%
1,Dead Hand (0 Supporter + 0 Energy),0.23%


In [14]:
print("=" * 50)
print("SPECIFIC CARD + SEARCHERS")
print("=" * 50)

total_target_copies = deck.quantity_of_names(TARGET_CARD_NAMES)
target_search_total = deck.quantity_of_names(TARGET_SEARCH_NAMES)

rows = []
for name in TARGET_CARD_NAMES:
    copies = deck.quantity_of(name)
    rows.append({
        "Statistic": f"P({name} in opening hand)",
        "Probability": f"{calc.specific_card_in_hand_probability(N, copies):.2%}",
    })

if len(TARGET_CARD_NAMES) > 1:
    rows.append({
        "Statistic": "P(any target in opening hand)",
        "Probability": f"{calc.specific_card_in_hand_probability(N, total_target_copies):.2%}",
    })

searcher_label = (
    f"P(Target search in hand) [{', '.join(TARGET_SEARCH_NAMES)}]"
    if TARGET_SEARCH_NAMES
    else "P(Target search in hand)"
)
rows.append({
    "Statistic": searcher_label,
    "Probability": f"{calc.searcher_probability(N, target_search_total):.2%}",
})
rows.append({
    "Statistic": "P(any target OR target search in hand)",
    "Probability": f"{calc.target_card_with_searches_probability(N, total_target_copies, target_search_total):.2%}",
})
display(pd.DataFrame(rows))

SPECIFIC CARD + SEARCHERS


,Statistic,Probability
0,P(N's Zorua in opening hand),0.00%
1,"P(Target search in hand) [Buddy-Buddy Poffin, ...",39.95%
2,P(any target OR target search in hand),39.95%


In [15]:
print("=" * 50)
print(f"MONTE CARLO VALIDATION (N={MC_SIMULATIONS:,} simulations)")
print("=" * 50)

print("Running simulation...")
sim = mc.simulate(deck, n=MC_SIMULATIONS, seed=MC_SEED)

# Compare key stats with theoretical values
comparison_data = [
    {
        "Statistic": "Mulligan rate",
        "Theoretical": f"{calc.mulligan_probability(N, b):.4f}",
        "Simulated": f"{sim['mulligan_rate']:.4f}",
        "Diff": f"{abs(sim['mulligan_rate'] - calc.mulligan_probability(N, b)):.4f}",
    },
    {
        "Statistic": "Supporter in hand",
        "Theoretical": f"{calc.supporter_turn1_probability(N, s):.4f}",
        "Simulated": f"{sim['supporter_in_hand']:.4f}",
        "Diff": f"{abs(sim['supporter_in_hand'] - calc.supporter_turn1_probability(N, s)):.4f}",
    },
    {
        "Statistic": "Dead hand",
        "Theoretical": f"{calc.dead_hand_probability(N, s, e):.4f}",
        "Simulated": f"{sim['dead_hand_rate']:.4f}",
        "Diff": f"{abs(sim['dead_hand_rate'] - calc.dead_hand_probability(N, s, e)):.4f}",
    },
]

# Add possible starters for each basic
for card in deck.basic_pokemon:
    theoretical = calc.possible_starter_probability(N, b, card.quantity)
    simulated = sim["possible_starters"].get(card.name, 0)
    comparison_data.append({
        "Statistic": f"Possible starter: {card.name}",
        "Theoretical": f"{theoretical:.4f}",
        "Simulated": f"{simulated:.4f}",
        "Diff": f"{abs(simulated - theoretical):.4f}",
    })

display(pd.DataFrame(comparison_data))
print("Monte Carlo validation complete.")


MONTE CARLO VALIDATION (N=100,000 simulations)
Running simulation...


,Statistic,Theoretical,Simulated,Diff
0,Mulligan rate,0.3991,0.3960,0.0031
1,Supporter in hand,0.9301,0.9280,0.0022
2,Dead hand,0.0023,0.0028,0.0005
3,Possible starter: Dwebble,0.6649,0.6658,0.0010
4,Possible starter: Mega Kangaskhan ex,0.3686,0.3697,0.0012
5,Possible starter: Cornerstone Mask Ogerpon ex,0.1942,0.1937,0.0005


Monte Carlo validation complete.
